In [1]:
from __future__ import annotations

import argparse
import re
import sys
import html
from html.parser import HTMLParser
from pathlib import Path
from typing import Iterable


# =============================================================================
# 【在这里粘贴】网页 HTML，或从浏览器复制的正文（纯文本/Markdown 均可）。
# 以 < 开头或含 <html/<body 的会按 HTML 解析；否则当纯文本。
#
# 默认把全文切成 NUM_CHUNKS 段（与命令行 --chunks 一致时可改此处）。
# Notebook：main(["--chunks", "9", "--out", "chunks.jsonl"])
# =============================================================================
NUM_CHUNKS = 7
SOURCE_TYPE = "return_policy"
DOCUMENT_NAME = "Apple"

PASTE_HTML_HERE = r"""
# Source: https://www.apple.com/shop/help/returns_refund

## SALES & REFUND TERMS AND CONDITIONS ("TERMS")

## U.S. Sales and Refund Policy

Thanks for shopping at Apple. We appreciate the fact that you like to buy the cool stuff we build. We also want to make sure you have a rewarding experience while you're exploring, evaluating, and purchasing our products, whether you're at the Apple Online Store, in an Apple Retail Store, or on the phone with the Apple Contact Center. (To make it visually easier on both of us, we'll refer to these entities as the "Apple Store" in this policy.)

As with any shopping experience, there are terms and conditions that apply to transactions at an Apple Store. We'll be as brief as our attorneys will allow. The main thing to remember is that by placing an order or making a purchase at an Apple Store, you agree to the terms set forth below along with Apple's Privacy Policy and Terms of Use.

## Standard Return Policy

We fundamentally believe you will be thrilled with the products you purchase from the Apple Store. That's because we go out of our way to ensure that they're designed and built to be just what you need. We understand, however, that sometimes a product may not be what you expected it to be. In that unlikely event, we invite you to review the following terms related to returning a product.

For any undamaged product, simply return it with its included accessories and packaging along with the original receipt (or gift receipt) within 14 days of the date you receive the product, and we'll exchange it or offer a refund based upon the original payment method. In addition, please note the following:

Products can be returned only in the country or region in which they were originally purchased.

The following products are not eligible for return: electronic software downloads, subscriptions to the Software-Up-To-Date program, Apple Store Gift Cards, and any Apple Developer Connection products.

For returns to an Apple Retail Store for cash, cash equivalent, and check transactions over $750, Apple will mail a refund check to you within 10 business days.

Should you wish to return ten or more of the same product, you must return to the Apple Store where originally purchased.

In the case of items returned with a gift receipt, Apple will offer you an Apple Gift Card.

If you paid for any portion of an item with your Apple Account balance, and are owed a refund for returning or cancelling it, we issue that portion of refund back to your Apple Account balance. However, we will issue this refund amount on an Apple Gift Card by email if your Apple Account balance is at or near the maximum limit.

Opened software cannot be returned if it contained a seal with the software license on the outside of the package and you could read the software license before opening its packaging. As an exception, you may return Apple-branded software if you do not agree to the licensing terms; however, you may not retain or otherwise use any copies of returned software.

Apple provides security features to enable you to protect your product in case of loss or theft. If these features have been activated and cannot be disabled by the person in possession of the phone, Apple may refuse the return or exchange.

For complete details on how to return a product purchased at the Apple Store please visit the Returns & Refunds page.

## Return of AppleCare+ under an iPhone Upgrade Program

Should you return the AppleCare+ portion of your iPhone Upgrade Program, please note that you will lose your Upgrade Option as set forth under the terms of the iPhone Upgrade Program.

## iPhone, iPad and Watch Returns - Wireless Service Cancellation

Wireless carriers have different service cancellation policies. Returning your iPhone, iPad or Watch may not automatically cancel or reset your wireless account; you are responsible for your wireless service agreement and for any applicable fees associated with your wireless account. Please contact your wireless service provider for more information.

## Apple Watch Returns

Apple Watch from the Edition collection may only be returned or exchanged if it's in its original, undamaged and unmarked condition after passing inspection at Apple's offsite facility. Depending on your original form of tender, a check, wire transfer, or refund to your debit/credit card will be issued within 10 business days provided the returned item is in its original condition.

## Additional Apple Product Terms

The purchase and use of Apple products are subject to additional terms and conditions found at https://www.apple.com/legal/sla/ and https://www.apple.com/legal/warranty/

Making unauthorized modifications to the software on an iPhone violates the iPhone software license agreement. The common term for modifying an iPhone is jail-breaking, with a particular emphasis on the second part of that term. That's why we strongly, almost emphatically, recommend that you do not do so. Really. Should you be unable to use your iPhone due to an unauthorized software modification, its repair will not be covered under the warranty.

## Pricing and Price Reductions/Corrections

Apple reserves the right to change prices for products displayed at/on the Apple Store at any time, and to correct pricing errors that may inadvertently occur. Additional information about pricing and sales tax is available on the Payment & Pricing page. In the event you have been charged more than the posted price for a product in an Apple Retail Store, please see a Manager for a refund of the overcharge.

Should Apple reduce its price on any Apple-branded product within 14 calendar days from the date you receive your product, feel free to visit an Apple Retail Store or contact the Apple Contact Center at 1-800-676-2775 to request a refund or credit of the difference between the price you were charged and the current selling price. To receive the refund or credit you must contact Apple within 14 calendar days of the price change. Please note that this excludes limited-time price reductions, such as those that occur during special sales events, such as Black Friday or Cyber Monday.

Price protection is only available for up to 10 units of a particular product. Additionally, we may require that you have the product with you or otherwise have proof of possession when requesting price protection.

Prices shown are in U.S. dollars. If you are paying for your order with an international Visa, MasterCard, or American Express credit card, please note that the purchase price may fluctuate with exchange rates. In addition, your bank or credit card issuer may also charge you foreign conversion charges and fees, which may also increase the overall cost of your purchase. Please contact your bank or credit card issuer regarding these fees.

## Order Acceptance/Confirmation

Apple may, in its sole discretion, refuse or cancel any order and limit order quantity. Apple may also require additional qualifying information prior to accepting or processing any order. Once we receive your Online or Call Center order, we'll provide you with an email order confirmation. Your receipt of an order confirmation, however, does not signify Apple's acceptance of your order, nor does it constitute confirmation of our offer to sell; we are simply confirming that we received your order. The Apple Store reserves the right at any time after receiving your order to accept or decline your order for any reason. If Apple cancels an order after you have already been billed, Apple will refund the billed amount.

## Shipping & Delivery

Please review the Shipping & Pickup page to learn about how and when you will receive the products you purchased from the Apple Store. Since the actual delivery of your order can be impacted by many events beyond Apple's control once it leaves our facilities, Apple cannot be held liable for late deliveries. We will, however, work with you to ensure a smooth delivery.

As Apple takes care of the dispatch of the products you purchase on the Apple Store, the risk of loss of, or damage to, product(s) shall pass to you when you, or a person designated by you, acquires physical possession of the product(s). Title in the product(s) shall pass to you when the product(s) is picked up by the carrier from our warehouse. At this point, you will receive the Shipment Notification Email. If there are any issues with delivery, please contact Apple to resolve.

## In-Store Pickup and Return

Apple offers in-store pickup for many of the items available on the Online Store. Certain products and payment methods, however, may not qualify for in-store pickup. Only you or the person designated by you may pick up the item(s) purchased. A government-issued photo ID and order number will be required for pickup. Apple will notify you when your order is ready and the date by which you need to pick up your items. We'll also send you a reminder or two, just in case it slips your mind. If you don't pick up your order, Apple may cancel it.

## Pickup Contact

If you select in-store pickup, you may designate a third party to pick up your order. You must provide the name and email address of the third party. Please note that certain products and payment methods are not eligible for in-store pickup by a third party. The third party will need to bring a government issued photo ID and order number for pickup. Apple is not responsible for actions taken by the third party once your item(s) have been picked up.

## Consumers Only

The Apple Store sells and ships products to end-user customers only, and we reserve the right to refuse or cancel your order if we suspect you are purchasing products for resale.

## U.S. Shipping Only

Products purchased online from Apple will only be shipped to addresses within the U.S. and are subject to U.S. and foreign export control laws and regulations. Products must be purchased, sold, exported, re-exported, transferred, and used in compliance with these export laws and regulations. To purchase Apple products online from outside of the U.S., please click here for international store information.

## Product Availability and Limitations

Given the popularity and/or supply constraints of some of our products, Apple may have to limit the number of products available for purchase. Trust us, we're building them as fast as we can. Apple reserves the right to change quantities available for purchase at any time, even after you place an order. Furthermore, there may be occasions when Apple confirms your order but subsequently learns that it cannot supply the ordered product. In the event we cannot supply a product you ordered, Apple will cancel the order and refund your purchase price in full.

## Gift Cards

For Apple Store Gift Card Terms and Conditions, please click here.
"""


_BLOCK_TAGS = frozenset(
    {
        "address",
        "article",
        "aside",
        "blockquote",
        "caption",
        "dd",
        "div",
        "dt",
        "figcaption",
        "figure",
        "footer",
        "form",
        "header",
        "hr",
        "li",
        "main",
        "nav",
        "p",
        "pre",
        "section",
        "table",
        "td",
        "th",
        "title",
        "tr",
    }
)
_SKIP_TAGS = frozenset({"script", "style", "noscript"})


class _HtmlToMarkdownish(HTMLParser):
    """HTML → plain text with # headings from h1–h6 (stdlib only)."""

    def __init__(self) -> None:
        super().__init__(convert_charrefs=True)
        self._parts: list[str] = []
        self._skip = 0
        self._h_level = 0

    def _emit(self, s: str) -> None:
        if s:
            self._parts.append(s)

    def handle_starttag(self, tag: str, attrs) -> None:
        t = tag.lower()
        if t in _SKIP_TAGS:
            self._skip += 1
            return
        if self._skip:
            return
        if t == "br":
            self._emit("\n")
            return
        if len(t) == 2 and t[0] == "h" and t[1].isdigit():
            level = int(t[1])
            if 1 <= level <= 6:
                self._h_level = level
                hashes = "#" * level
                self._emit("\n\n" + hashes + " ")
            return
        if t in _BLOCK_TAGS:
            self._emit("\n")

    def handle_endtag(self, tag: str) -> None:
        t = tag.lower()
        if t in _SKIP_TAGS:
            self._skip = max(0, self._skip - 1)
            return
        if self._skip:
            return
        if len(t) == 2 and t[0] == "h" and t[1].isdigit():
            level = int(t[1])
            if level == self._h_level:
                self._h_level = 0
                self._emit("\n")
            return
        if t in _BLOCK_TAGS:
            self._emit("\n")

    def handle_data(self, data: str) -> None:
        if self._skip:
            return
        text = html.unescape(data).replace("\u00a0", " ")
        if self._h_level:
            text = " ".join(text.split())
        self._emit(text)

    def handle_entityref(self, name: str) -> None:
        if self._skip:
            return
        self._emit(html.unescape(f"&{name};"))

    def handle_charref(self, name: str) -> None:
        if self._skip:
            return
        self._emit(html.unescape(f"&#{name};"))

    def text(self) -> str:
        raw = "".join(self._parts)
        raw = re.sub(r"[ \t]+\n", "\n", raw)
        raw = re.sub(r"\n{3,}", "\n\n", raw)
        return raw.strip()


def html_to_chunkable_text(html: str) -> str:
    parser = _HtmlToMarkdownish()
    parser.feed(html)
    parser.close()
    return parser.text()


def normalize_pasted_source(pasted: str) -> str:
    """HTML → text; plain text / Markdown left as-is."""
    s = pasted.strip()
    probe = s[:1200].lstrip()
    low = probe.lower()
    if probe.startswith("<") or "<html" in low or "<body" in low:
        return html_to_chunkable_text(pasted)
    return s


def read_pdf(path: Path) -> str:
    try:
        from pypdf import PdfReader
    except ImportError as e:
        raise SystemExit(
            "Reading PDF requires pypdf. Install with: pip install pypdf"
        ) from e
    reader = PdfReader(str(path))
    parts: list[str] = []
    for page in reader.pages:
        t = page.extract_text() or ""
        parts.append(t)
    return "\n".join(parts)


def read_document(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        return read_pdf(path)
    return path.read_text(encoding="utf-8", errors="replace")


def split_full_text_into_n(text: str, n: int) -> list[str]:
    """把 ``text`` strip 后按字符下标均分成 ``n`` 段（最后几段可能因整除差 1 个字符）。"""
    text = text.strip()
    if n < 1:
        n = 1
    if not text:
        return [""] * n
    L = len(text)
    bounds = [(L * k) // n for k in range(n + 1)]
    return [text[bounds[k] : bounds[k + 1]] for k in range(n)]


def build_chunks(text: str, *, num_chunks: int) -> list[dict]:
    """``num_chunks`` 条：chunk_id, text, source_type, document_name。"""
    pieces = split_full_text_into_n(text, num_chunks)
    return [
        {
            "chunk_id": i + 1,
            "text": p,
            "source_type": SOURCE_TYPE,
            "document_name": DOCUMENT_NAME,
        }
        for i, p in enumerate(pieces)
    ]


def iter_print_chunks(chunks: Iterable[dict], *, print_lengths: bool = False) -> None:
    for c in chunks:
        print("-" * 60)
        print(
            f"Chunk {c['chunk_id']} | {c['source_type']} | {c['document_name']}"
        )
        print(c["text"])
        if print_lengths:
            body = c["text"]
            n_all = len(body)
            n_ns = len(re.sub(r"\s+", "", body))
            print(f"# len(all)={n_all} len(non-space)={n_ns}")
        print()


def strip_jupyter_kernel_argv(argv: list[str]) -> list[str]:
    """Drop ipykernel's ``-f <connection.json>`` so it is not parsed as ``path``."""
    out: list[str] = []
    i = 0
    while i < len(argv):
        a = argv[i]
        if a == "-f" and i + 1 < len(argv):
            i += 2
            continue
        if a.startswith("-f") and len(a) > 2:
            i += 1
            continue
        out.append(a)
        i += 1
    return out


def main(argv: list[str] | None = None) -> int:
    p = argparse.ArgumentParser(description="Chunk text for LLM prompts.")
    p.add_argument(
        "path",
        nargs="?",
        type=Path,
        default=None,
        help="Path to .txt, .md, or .pdf. Omit to use PASTE_HTML_HERE in this file.",
    )
    p.add_argument(
        "--chunks",
        type=int,
        default=NUM_CHUNKS,
        help=f"Split full document into this many equal parts (default {NUM_CHUNKS}).",
    )
    p.add_argument(
        "--print-lengths",
        action="store_true",
        help="After each chunk, print len(all) and len(non-space) for debugging.",
    )
    p.add_argument(
        "--out",
        type=Path,
        default=None,
        help="Optional JSONL output path (one JSON object per line).",
    )
    # Jupyter：argv 常为 ``-f kernel-....json``。须整段去掉 -f 及其值，否则 json 路径会被
    # 当成输入文件（你就会看到 shell_port / kernel_name 被切成 chunk）。
    if argv is None:
        argv = strip_jupyter_kernel_argv(sys.argv[1:])
    else:
        argv = strip_jupyter_kernel_argv(list(argv))
    args, _unknown = p.parse_known_args(argv)

    path: Path | None = args.path
    if path is None:
        pasted = PASTE_HTML_HERE.strip()
        if not pasted:
            print(
                "未指定文件，且 PASTE_HTML_HERE 为空。\n"
                "请在 text_chunker.py 里找到「【在这里粘贴】」"
                "，把内容粘进三引号之间；或运行： python text_chunker.py yourfile.md",
                file=sys.stderr,
            )
            return 1
        raw = normalize_pasted_source(pasted)
    else:
        if not path.is_file():
            print(f"File not found: {path}", file=sys.stderr)
            return 1
        raw = read_document(path)

    if args.chunks < 1:
        print("--chunks must be >= 1.", file=sys.stderr)
        return 1

    chunks = build_chunks(raw, num_chunks=args.chunks)

    if args.out:
        import json

        with args.out.open("w", encoding="utf-8") as f:
            for c in chunks:
                f.write(json.dumps(c, ensure_ascii=False) + "\n")
        print(f"Wrote {len(chunks)} chunks to {args.out}", file=sys.stderr)
    else:
        iter_print_chunks(chunks, print_lengths=args.print_lengths)

    return 0


if __name__ == "__main__":
    # 成功时不要用 raise SystemExit(0)：在 IPython / Jupyter 里 %run 会触发
    # "To exit: use 'exit', 'quit', or Ctrl-D." 的 UserWarning。
    _code = main()
    if _code:
        sys.exit(_code)

------------------------------------------------------------
Chunk 1 | return_policy | Apple
# Source: https://www.apple.com/shop/help/returns_refund

## SALES & REFUND TERMS AND CONDITIONS ("TERMS")

## U.S. Sales and Refund Policy

Thanks for shopping at Apple. We appreciate the fact that you like to buy the cool stuff we build. We also want to make sure you have a rewarding experience while you're exploring, evaluating, and purchasing our products, whether you're at the Apple Online Store, in an Apple Retail Store, or on the phone with the Apple Contact Center. (To make it visually easier on both of us, we'll refer to these entities as the "Apple Store" in this policy.)

As with any shopping experience, there are terms and conditions that apply to transactions at an Apple Store. We'll be as brief as our attorneys will allow. The main thing to remember is that by placing an order or making a purchase at an Apple Store, you agree to the terms set forth below along with Apple's Privacy

In [3]:
NUM_CHUNKS = 3
CHUNK_DISPLAY_START = 8
SOURCE_TYPE = "return_policy"
DOCUMENT_NAME = "Aritzia"

PASTE_HTML_HERE = r"""
# Source: https://www.aritzia.com/us/en/returns

# Return and Exchange Policy

At Aritzia, clients are at the heart of what we do. We aim to be responsive and responsible as we operate with safety in mind. Returns are free within the United States when you are signed into your Aritzia account, subject to the rules below.

## Online purchases

For purchases made on aritzia.com, you can return online and in store. Exchanges for online orders may be completed in a boutique or by mailing back your return and placing a new order. Follow the mail-back instructions on the site when you start a return from your account or order tracking.

### Regular priced merchandise

Returns: Within fourteen days from the shipping date, you can receive a full refund to the original method of payment. Within thirty days from the shipping date, you can exchange your item or return for merchandise credit on a Digital Gift Card. After thirty days, returns are no longer accepted.

Exchanges: Online purchases can be exchanged in store and through Concierge within thirty days from the shipping date. For help with online exchanges, start a live chat, call 1-855-ARITZIA, or email Concierge.

### Sale merchandise

Within fourteen days from the shipping date, sale items marked with a discount less than fifty percent can be exchanged or returned for merchandise credit according to the posted rules. Sale items marked at fifty percent off or greater are final sale unless stated otherwise on the product page.

## In store purchases

Within fourteen days of purchase, you can return your item and receive a refund to the original method of payment at any boutique in the same country where you bought the item. Within thirty days of purchase, you can exchange your item or receive merchandise credit at any boutique in the same country. After thirty days, returns are not accepted.

Sale items with a discount under fifty percent may be exchanged or returned for merchandise credit within fourteen days at a boutique in the same country. Sale items at fifty percent off or greater are final sale in store as well.

## Return and exchange conditions

For all returns or exchanges, items must not be washed, worn, or damaged. All original tags must be attached. You must present a valid receipt or proof of purchase. Swimwear liner must be attached where applicable. Shoes must be returned in the original box.

Final sale and not eligible for return or exchange unless required by law can include sale items marked at fifty percent off or greater, intimates and shapewear, hosiery, face masks, and certain personalized or special products as listed on the site.

Gift cards may not be redeemed for cash or refunded except where required by law. If you used a gift card or third-party pay-in-four products, refunds follow the same policy when your return qualifies and may be returned to the original tender where possible.

Aritzia may update this policy. Check aritzia.com for the current Return and Exchange Policy before you shop.
"""


_BLOCK_TAGS = frozenset(
    {
        "address",
        "article",
        "aside",
        "blockquote",
        "caption",
        "dd",
        "div",
        "dt",
        "figcaption",
        "figure",
        "footer",
        "form",
        "header",
        "hr",
        "li",
        "main",
        "nav",
        "p",
        "pre",
        "section",
        "table",
        "td",
        "th",
        "title",
        "tr",
    }
)
_SKIP_TAGS = frozenset({"script", "style", "noscript"})


class _HtmlToMarkdownish(HTMLParser):
    """HTML → plain text with # headings from h1–h6 (stdlib only)."""

    def __init__(self) -> None:
        super().__init__(convert_charrefs=True)
        self._parts: list[str] = []
        self._skip = 0
        self._h_level = 0

    def _emit(self, s: str) -> None:
        if s:
            self._parts.append(s)

    def handle_starttag(self, tag: str, attrs) -> None:
        t = tag.lower()
        if t in _SKIP_TAGS:
            self._skip += 1
            return
        if self._skip:
            return
        if t == "br":
            self._emit("\n")
            return
        if len(t) == 2 and t[0] == "h" and t[1].isdigit():
            level = int(t[1])
            if 1 <= level <= 6:
                self._h_level = level
                hashes = "#" * level
                self._emit("\n\n" + hashes + " ")
            return
        if t in _BLOCK_TAGS:
            self._emit("\n")

    def handle_endtag(self, tag: str) -> None:
        t = tag.lower()
        if t in _SKIP_TAGS:
            self._skip = max(0, self._skip - 1)
            return
        if self._skip:
            return
        if len(t) == 2 and t[0] == "h" and t[1].isdigit():
            level = int(t[1])
            if level == self._h_level:
                self._h_level = 0
                self._emit("\n")
            return
        if t in _BLOCK_TAGS:
            self._emit("\n")

    def handle_data(self, data: str) -> None:
        if self._skip:
            return
        text = html.unescape(data).replace("\u00a0", " ")
        if self._h_level:
            text = " ".join(text.split())
        self._emit(text)

    def handle_entityref(self, name: str) -> None:
        if self._skip:
            return
        self._emit(html.unescape(f"&{name};"))

    def handle_charref(self, name: str) -> None:
        if self._skip:
            return
        self._emit(html.unescape(f"&#{name};"))

    def text(self) -> str:
        raw = "".join(self._parts)
        raw = re.sub(r"[ \t]+\n", "\n", raw)
        raw = re.sub(r"\n{3,}", "\n\n", raw)
        return raw.strip()


def html_to_chunkable_text(html: str) -> str:
    parser = _HtmlToMarkdownish()
    parser.feed(html)
    parser.close()
    return parser.text()


def normalize_pasted_source(pasted: str) -> str:
    """HTML → text; plain text / Markdown left as-is."""
    s = pasted.strip()
    probe = s[:1200].lstrip()
    low = probe.lower()
    if probe.startswith("<") or "<html" in low or "<body" in low:
        return html_to_chunkable_text(pasted)
    return s


def read_pdf(path: Path) -> str:
    try:
        from pypdf import PdfReader
    except ImportError as e:
        raise SystemExit(
            "Reading PDF requires pypdf. Install with: pip install pypdf"
        ) from e
    reader = PdfReader(str(path))
    parts: list[str] = []
    for page in reader.pages:
        t = page.extract_text() or ""
        parts.append(t)
    return "\n".join(parts)


def read_document(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        return read_pdf(path)
    return path.read_text(encoding="utf-8", errors="replace")


def split_full_text_into_n(text: str, n: int) -> list[str]:
    """按 ``str.split()`` 整词切分；``n`` 块内词数尽量均匀，绝不截断单词。"""
    text = text.strip()
    if n < 1:
        n = 1
    if not text:
        return [""] * n
    words = text.split()
    if not words:
        return [""] * n
    if n == 1:
        return [text]
    W = len(words)
    # 第 i 块词数：floor 均分，余数 r 由前 r 块各多 1 个词（W<n 时自然出现若干空块）
    sizes = [W // n + (1 if i < W % n else 0) for i in range(n)]
    idx = 0
    out: list[str] = []
    for sz in sizes:
        out.append(" ".join(words[idx : idx + sz]))
        idx += sz
    return out


def build_chunks(text: str, *, num_chunks: int) -> list[dict]:
    """``num_chunks`` 条：chunk_id（从 ``CHUNK_DISPLAY_START`` 递增）, text, source_type, document_name。"""
    pieces = split_full_text_into_n(text, num_chunks)
    return [
        {
            "chunk_id": CHUNK_DISPLAY_START + i,
            "text": p,
            "source_type": SOURCE_TYPE,
            "document_name": DOCUMENT_NAME,
        }
        for i, p in enumerate(pieces)
    ]


def iter_print_chunks(chunks: Iterable[dict], *, print_lengths: bool = False) -> None:
    for c in chunks:
        print("-" * 60)
        print(
            f"Chunk {c['chunk_id']} | {c['source_type']} | {c['document_name']}"
        )
        print(c["text"])
        if print_lengths:
            body = c["text"]
            n_all = len(body)
            n_ns = len(re.sub(r"\s+", "", body))
            print(f"# len(all)={n_all} len(non-space)={n_ns}")
        print()


def strip_jupyter_kernel_argv(argv: list[str]) -> list[str]:
    """Drop ipykernel's ``-f <connection.json>`` so it is not parsed as ``path``."""
    out: list[str] = []
    i = 0
    while i < len(argv):
        a = argv[i]
        if a == "-f" and i + 1 < len(argv):
            i += 2
            continue
        if a.startswith("-f") and len(a) > 2:
            i += 1
            continue
        out.append(a)
        i += 1
    return out


def main(argv: list[str] | None = None) -> int:
    p = argparse.ArgumentParser(description="Chunk text for LLM prompts.")
    p.add_argument(
        "path",
        nargs="?",
        type=Path,
        default=None,
        help="Path to .txt, .md, or .pdf. Omit to use PASTE_HTML_HERE in this file.",
    )
    p.add_argument(
        "--chunks",
        type=int,
        default=NUM_CHUNKS,
        help=(
            f"Split into this many chunks by whole words (default {NUM_CHUNKS}); "
            "does not break mid-token."
        ),
    )
    p.add_argument(
        "--print-lengths",
        action="store_true",
        help="After each chunk, print len(all) and len(non-space) for debugging.",
    )
    p.add_argument(
        "--out",
        type=Path,
        default=None,
        help="Optional JSONL output path (one JSON object per line).",
    )
    # Jupyter：argv 常为 ``-f kernel-....json``。须整段去掉 -f 及其值，否则 json 路径会被
    # 当成输入文件（你就会看到 shell_port / kernel_name 被切成 chunk）。
    if argv is None:
        argv = strip_jupyter_kernel_argv(sys.argv[1:])
    else:
        argv = strip_jupyter_kernel_argv(list(argv))
    args, _unknown = p.parse_known_args(argv)

    path: Path | None = args.path
    if path is None:
        pasted = PASTE_HTML_HERE.strip()
        if not pasted:
            print(
                "未指定文件，且 PASTE_HTML_HERE 为空。\n"
                "请在 text_chunker.py 里找到「【在这里粘贴】」"
                "，把内容粘进三引号之间；或运行： python text_chunker.py yourfile.md",
                file=sys.stderr,
            )
            return 1
        raw = normalize_pasted_source(pasted)
    else:
        if not path.is_file():
            print(f"File not found: {path}", file=sys.stderr)
            return 1
        raw = read_document(path)

    if args.chunks < 1:
        print("--chunks must be >= 1.", file=sys.stderr)
        return 1

    chunks = build_chunks(raw, num_chunks=args.chunks)

    if args.out:
        import json

        with args.out.open("w", encoding="utf-8") as f:
            for c in chunks:
                f.write(json.dumps(c, ensure_ascii=False) + "\n")
        print(f"Wrote {len(chunks)} chunks to {args.out}", file=sys.stderr)
    else:
        iter_print_chunks(chunks, print_lengths=args.print_lengths)

    return 0


if __name__ == "__main__":
    # 成功时不要用 raise SystemExit(0)：在 IPython / Jupyter 里 %run 会触发
    # "To exit: use 'exit', 'quit', or Ctrl-D." 的 UserWarning。
    _code = main()
    if _code:
        sys.exit(_code)

register_chunks(chunks, source_label="html_aritzia")

------------------------------------------------------------
Chunk 8 | return_policy | Aritzia
# Source: https://www.aritzia.com/us/en/returns # Return and Exchange Policy At Aritzia, clients are at the heart of what we do. We aim to be responsive and responsible as we operate with safety in mind. Returns are free within the United States when you are signed into your Aritzia account, subject to the rules below. ## Online purchases For purchases made on aritzia.com, you can return online and in store. Exchanges for online orders may be completed in a boutique or by mailing back your return and placing a new order. Follow the mail-back instructions on the site when you start a return from your account or order tracking. ### Regular priced merchandise Returns: Within fourteen days from the shipping date, you can receive a full refund to the original method of payment. Within thirty days from the shipping date, you can exchange your item or return for merchandise credit on a Digital Gift 

NameError: name 'register_chunks' is not defined

In [5]:
NUM_CHUNKS = 6
CHUNK_DISPLAY_START = 11
SOURCE_TYPE = "syllabus"
DOCUMENT_NAME = "561"

# PDF：默认桌面「ISE 547」文件夹里的 558.pdf；若路径不同请改成你的绝对路径
PDF_PATH = Path.home() / "Desktop" / "ISE 547" / "561.pdf"

NUM_CHUNKS_OVERRIDE: int | None = None  # 填 None 用上面的 NUM_CHUNKS
PDF_ENGINE: str = "auto"  # "auto" | "pypdf" | "pdfplumber"
# 不抽取 PDF 最后若干页（例如附录、空白页）；0 = 读全文
SKIP_LAST_PAGES: int = 3

OUT_JSONL: Path | None = None  # None = 打印；或 Path("chunks.jsonl")


# ----- PDF 与切块（原 text_chunker 里与 PDF/切块相关的最小子集）----------------


def _pages_without_tail(page_list: list, *, skip_last: int) -> list:
    """skip_last > 0 时去掉末尾 skip_last 页；总页数不足时只保留前面能保留的页（可能为空）。"""
    if skip_last <= 0:
        return page_list
    n = len(page_list)
    if n <= skip_last:
        return []
    return page_list[: n - skip_last]


def _read_pdf_text_pypdf(path: Path, *, skip_last_pages: int) -> str:
    from pypdf import PdfReader

    reader = PdfReader(str(path))
    pages = list(reader.pages)
    pages = _pages_without_tail(pages, skip_last=skip_last_pages)
    parts: list[str] = []
    for page in pages:
        parts.append(page.extract_text() or "")
    return "\n".join(parts)


def _read_pdf_text_pdfplumber(path: Path, *, skip_last_pages: int) -> str:
    import pdfplumber

    parts: list[str] = []
    with pdfplumber.open(str(path)) as pdf:
        pages = _pages_without_tail(list(pdf.pages), skip_last=skip_last_pages)
        for page in pages:
            parts.append(page.extract_text() or "")
    return "\n".join(parts)


def read_pdf_text(
    path: str | Path, *, engine: str = "auto", skip_last_pages: int = 0
) -> str:
    p = Path(path)
    eng = engine.strip().lower()
    if eng == "pypdf":
        return _read_pdf_text_pypdf(p, skip_last_pages=skip_last_pages)
    if eng == "pdfplumber":
        return _read_pdf_text_pdfplumber(p, skip_last_pages=skip_last_pages)
    if eng == "auto":
        try:
            return _read_pdf_text_pdfplumber(p, skip_last_pages=skip_last_pages)
        except ImportError:
            try:
                return _read_pdf_text_pypdf(p, skip_last_pages=skip_last_pages)
            except ImportError as e2:
                raise ImportError(
                    "读 PDF 需要安装 pypdf 和/或 pdfplumber：pip install pypdf pdfplumber"
                ) from e2
    raise ValueError(f"engine 必须是 pypdf|pdfplumber|auto，收到：{engine!r}")


def split_full_text_into_n(text: str, n: int) -> list[str]:
    text = text.strip()
    if n < 1:
        n = 1
    if not text:
        return [""] * n
    words = text.split()
    if not words:
        return [""] * n
    if n == 1:
        return [text]
    W = len(words)
    sizes = [W // n + (1 if i < W % n else 0) for i in range(n)]
    idx = 0
    out: list[str] = []
    for sz in sizes:
        out.append(" ".join(words[idx : idx + sz]))
        idx += sz
    return out


def build_chunks(text: str, *, num_chunks: int) -> list[dict]:
    pieces = split_full_text_into_n(text, num_chunks)
    return [
        {
            "chunk_id": CHUNK_DISPLAY_START + i,
            "text": p,
            "source_type": SOURCE_TYPE,
            "document_name": DOCUMENT_NAME,
        }
        for i, p in enumerate(pieces)
    ]


def iter_print_chunks(chunks: Iterable[dict]) -> None:
    for c in chunks:
        print("-" * 60)
        print(f"Chunk {c['chunk_id']} | {c['source_type']} | {c['document_name']}")
        print(c["text"])
        print()


def main() -> int:
    if not PDF_PATH.is_file():
        print(f"[错误] 找不到 PDF：{PDF_PATH.resolve()}", flush=True)
        print("请修改本文件里的 PDF_PATH（桌面文件夹名或文件名若不同也要改）。", flush=True)
        return 1

    raw = read_pdf_text(
        PDF_PATH, engine=PDF_ENGINE, skip_last_pages=SKIP_LAST_PAGES
    )
    n = NUM_CHUNKS_OVERRIDE if NUM_CHUNKS_OVERRIDE is not None else NUM_CHUNKS
    chunks = build_chunks(raw, num_chunks=n)

    if OUT_JSONL is not None:
        with OUT_JSONL.open("w", encoding="utf-8") as f:
            for c in chunks:
                f.write(json.dumps(c, ensure_ascii=False) + "\n")
        print(f"[完成] 已写入 {len(chunks)} 条 → {OUT_JSONL.resolve()}", flush=True)
    else:
        iter_print_chunks(chunks)

    return 0


if __name__ == "__main__":
    _code = main()
    if _code:
        sys.exit(_code)


------------------------------------------------------------
Chunk 11 | syllabus | 561
ISE561-Economic Analysis of Engineering Projects Units: 4 Fall 2025 Tuesday/Thursday 10:00 am- 11:50 am Location: RTH 109 http://courses.uscden.net Instructor: Dr. Shalini Gupta Contact Info: shalinig@usc.edu Office Hours: Thursday 4:30 pm – 5:30 pm (Via zoom) Teaching Assistant: TBD Course Description Economic evaluations of engineering systems for both government and private industry; quantitative techniques for evaluating non-monetary consequences; formal treatment of risk and uncertainty. Student Learning Outcomes: Students will be able to: • determine the equivalent value of money at a specified time given the timing of deposits and interest value; • select the most attractive interest rate in various compound and simple interest forms; • determine if an independent investment opportunity is economically attractive; • determine the least-cost alternative of multiple solutions in a cost compariso

In [9]:
NUM_CHUNKS = 9
CHUNK_DISPLAY_START = 17
SOURCE_TYPE = "syllabus"
DOCUMENT_NAME = "534"

# PDF：默认桌面「ISE 547」文件夹里的 558.pdf；若路径不同请改成你的绝对路径
PDF_PATH = Path.home() / "Desktop" / "ISE 547" / "534.pdf"

NUM_CHUNKS_OVERRIDE: int | None = None  # 填 None 用上面的 NUM_CHUNKS
PDF_ENGINE: str = "auto"  # "auto" | "pypdf" | "pdfplumber"
# 不抽取 PDF 最后若干页（例如附录、空白页）；0 = 读全文
SKIP_LAST_PAGES: int = 2

OUT_JSONL: Path | None = None  # None = 打印；或 Path("chunks.jsonl")


# ----- PDF 与切块（原 text_chunker 里与 PDF/切块相关的最小子集）----------------


def _pages_without_tail(page_list: list, *, skip_last: int) -> list:
    """skip_last > 0 时去掉末尾 skip_last 页；总页数不足时只保留前面能保留的页（可能为空）。"""
    if skip_last <= 0:
        return page_list
    n = len(page_list)
    if n <= skip_last:
        return []
    return page_list[: n - skip_last]


def _read_pdf_text_pypdf(path: Path, *, skip_last_pages: int) -> str:
    from pypdf import PdfReader

    reader = PdfReader(str(path))
    pages = list(reader.pages)
    pages = _pages_without_tail(pages, skip_last=skip_last_pages)
    parts: list[str] = []
    for page in pages:
        parts.append(page.extract_text() or "")
    return "\n".join(parts)


def _read_pdf_text_pdfplumber(path: Path, *, skip_last_pages: int) -> str:
    import pdfplumber

    parts: list[str] = []
    with pdfplumber.open(str(path)) as pdf:
        pages = _pages_without_tail(list(pdf.pages), skip_last=skip_last_pages)
        for page in pages:
            parts.append(page.extract_text() or "")
    return "\n".join(parts)


def read_pdf_text(
    path: str | Path, *, engine: str = "auto", skip_last_pages: int = 0
) -> str:
    p = Path(path)
    eng = engine.strip().lower()
    if eng == "pypdf":
        return _read_pdf_text_pypdf(p, skip_last_pages=skip_last_pages)
    if eng == "pdfplumber":
        return _read_pdf_text_pdfplumber(p, skip_last_pages=skip_last_pages)
    if eng == "auto":
        try:
            return _read_pdf_text_pdfplumber(p, skip_last_pages=skip_last_pages)
        except ImportError:
            try:
                return _read_pdf_text_pypdf(p, skip_last_pages=skip_last_pages)
            except ImportError as e2:
                raise ImportError(
                    "读 PDF 需要安装 pypdf 和/或 pdfplumber：pip install pypdf pdfplumber"
                ) from e2
    raise ValueError(f"engine 必须是 pypdf|pdfplumber|auto，收到：{engine!r}")


def split_full_text_into_n(text: str, n: int) -> list[str]:
    text = text.strip()
    if n < 1:
        n = 1
    if not text:
        return [""] * n
    words = text.split()
    if not words:
        return [""] * n
    if n == 1:
        return [text]
    W = len(words)
    sizes = [W // n + (1 if i < W % n else 0) for i in range(n)]
    idx = 0
    out: list[str] = []
    for sz in sizes:
        out.append(" ".join(words[idx : idx + sz]))
        idx += sz
    return out


def build_chunks(text: str, *, num_chunks: int) -> list[dict]:
    pieces = split_full_text_into_n(text, num_chunks)
    return [
        {
            "chunk_id": CHUNK_DISPLAY_START + i,
            "text": p,
            "source_type": SOURCE_TYPE,
            "document_name": DOCUMENT_NAME,
        }
        for i, p in enumerate(pieces)
    ]


def iter_print_chunks(chunks: Iterable[dict]) -> None:
    for c in chunks:
        print("-" * 60)
        print(f"Chunk {c['chunk_id']} | {c['source_type']} | {c['document_name']}")
        print(c["text"])
        print()


def main() -> int:
    if not PDF_PATH.is_file():
        print(f"[错误] 找不到 PDF：{PDF_PATH.resolve()}", flush=True)
        print("请修改本文件里的 PDF_PATH（桌面文件夹名或文件名若不同也要改）。", flush=True)
        return 1

    raw = read_pdf_text(
        PDF_PATH, engine=PDF_ENGINE, skip_last_pages=SKIP_LAST_PAGES
    )
    n = NUM_CHUNKS_OVERRIDE if NUM_CHUNKS_OVERRIDE is not None else NUM_CHUNKS
    chunks = build_chunks(raw, num_chunks=n)

    if OUT_JSONL is not None:
        with OUT_JSONL.open("w", encoding="utf-8") as f:
            for c in chunks:
                f.write(json.dumps(c, ensure_ascii=False) + "\n")
        print(f"[完成] 已写入 {len(chunks)} 条 → {OUT_JSONL.resolve()}", flush=True)
    else:
        iter_print_chunks(chunks)

    return 0


if __name__ == "__main__":
    _code = main()
    if _code:
        sys.exit(_code)


------------------------------------------------------------
Chunk 17 | syllabus | 534
ISE 534 - Data Analytics Consulting Time: One Lectures /Week Units: 4.0 Instructor: Saeed (SID) Mohasseb Office: N/A Office Hours: 3:00 to 4:00 on Mondays with prior appointment PLUS: Special One on One or team remote automated calendaring system: https://mohasseb.as.me/schedule.php Contact Info: Email: Sid@Mohasseb.com Cell: 949-254-9280 General timeline to respond to emails & calls: within 48 hours. Class Location” SLH 100 - Stauffer Science Lecture Hall Teaching Assistant: TBD Office Hours: TBD Contact Info: TBD Course Description Consulting project concepts, frameworks, analytical tools, and managerial skills with a focus on the use of data analytics, design thinking and insight-driven frameworks. Course Overview The course offers hands on project centric and experiential learning in using technical data analytics elements and business and execution factors. The course uses an industry driven app

In [11]:
import json
import re
import sys
from pathlib import Path
from typing import Iterable

In [13]:
PDF_PATH = Path.home() / "Desktop" / "ISE 547" / "Americas-AI-Action-Plan.pdf" # 改成你的 PDF 绝对路径
PDF_ENGINE: str = "auto"  # "auto" | "pypdf" | "pdfplumber"

# 不读取、不写入任何 chunk 的尾部页数（相对 PDF 末尾）
SKIP_LAST_PAGES: int = 2

# 正文从第几页开始才计入 chunk（含）。第 4 页起 = 不包含第 1–3 页正文
CONTENT_FIRST_PAGE_1BASED: int = 4

# 若正文首行与目录标题（去空白后）相同，去掉该行以免重复
DEDUPE_TITLE_FIRST_LINE_IN_BODY: bool = True

OUT_JSONL: Path | None = None

TOC_CHUNKS_START_ID = 26
TOC_SOURCE_TYPE = "macro_policy"
TOC_DOCUMENT_NAME = "Action_plan"
TOC_PARSE_PAGE_START = 3
# 目录有时跨两页，若只解析到十几条，改为 4 或把下面 TOC_DEBUG 打开看原文
TOC_PARSE_PAGE_END = 3

# 为 True 时在 stderr 打印：目录页字符数、解析到的条目数、前几条标题（排查「只有十几 chunk」）
TOC_DEBUG: bool = False

# 点状指引线：半角点、省略号、或「点+空格」混排（PDF 常见）
_TOC_LEADER = r"(?:\.{3,}|…{2,}|(?:\.\s*){3,})"
_TOC_ITEM_RE = re.compile(
    rf"(?P<title>.+?)\s*{_TOC_LEADER}\s*(?P<page>\d{{1,4}})\b",
    re.UNICODE,
)
# 无点、仅用大空白把标题和页码分开的行
_TOC_SPACED_RE = re.compile(r"^(.+?)\s{4,}(\d{1,4})\s*$")


def _pages_without_tail(page_list: list, *, skip_last: int) -> list:
    if skip_last <= 0:
        return page_list
    n = len(page_list)
    if n <= skip_last:
        return []
    return page_list[: n - skip_last]


def _read_pdf_pages_pypdf(path: Path, *, skip_last_pages: int) -> list[str]:
    from pypdf import PdfReader

    reader = PdfReader(str(path))
    pages = _pages_without_tail(list(reader.pages), skip_last=skip_last_pages)
    return [(p.extract_text() or "") for p in pages]


def _read_pdf_pages_pdfplumber(path: Path, *, skip_last_pages: int) -> list[str]:
    import pdfplumber

    with pdfplumber.open(str(path)) as pdf:
        pages = _pages_without_tail(list(pdf.pages), skip_last=skip_last_pages)
        return [(p.extract_text() or "") for p in pages]


def read_pdf_pages(
    path: str | Path, *, engine: str = "auto", skip_last_pages: int = 0
) -> list[str]:
    p = Path(path)
    eng = engine.strip().lower()
    if eng == "pypdf":
        return _read_pdf_pages_pypdf(p, skip_last_pages=skip_last_pages)
    if eng == "pdfplumber":
        return _read_pdf_pages_pdfplumber(p, skip_last_pages=skip_last_pages)
    if eng == "auto":
        try:
            return _read_pdf_pages_pdfplumber(p, skip_last_pages=skip_last_pages)
        except ImportError:
            try:
                return _read_pdf_pages_pypdf(p, skip_last_pages=skip_last_pages)
            except ImportError as e2:
                raise ImportError(
                    "读 PDF 需要安装 pypdf 和/或 pdfplumber：pip install pypdf pdfplumber"
                ) from e2
    raise ValueError(f"engine 必须是 pypdf|pdfplumber|auto，收到：{engine!r}")


def _toc_pages_text(pages: list[str], start_1: int, end_1: int) -> str:
    n = len(pages)
    if n == 0:
        return ""
    a = max(1, start_1) - 1
    b = min(n, end_1)
    if a >= b:
        return ""
    return "\n".join(pages[a:b])


def parse_toc_entries(toc_blob: str) -> list[tuple[str, int]]:
    """从目录页文本抽出 (标题, 页码)。**同一行可有多条**（finditer）；**不按页合并**（避免 30 条变十几条）。"""
    entries: list[tuple[str, int]] = []
    pending_title: str | None = None

    skip_titles = frozenset({"contents", "table of contents"})

    def push(title: str, page: int) -> None:
        t = " ".join(title.split())
        t = re.sub(r"[\.…\s]+$", "", t).strip()
        if len(t) < 3 or page < 1 or page > 9999:
            return
        if t.casefold() in skip_titles:
            return
        entries.append((t, page))

    for raw_line in toc_blob.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        # 标题在上一行、页码单独一行（换行抽乱时偶发）
        if pending_title and re.fullmatch(r"\d{1,4}", line):
            push(pending_title, int(line))
            pending_title = None
            continue

        found = list(_TOC_ITEM_RE.finditer(line))
        if found:
            pending_title = None
            for m in found:
                push(m.group("title"), int(m.group("page")))
            continue

        m2 = _TOC_SPACED_RE.match(line)
        if m2:
            pending_title = None
            push(m2.group(1), int(m2.group(2)))
            continue

        # 可能是折行标题（下一行才是 …… 页码）——仅在尚无 pending 时收一行，避免乱吞正文
        if pending_title is None and len(line) > 12 and re.search(r"[A-Za-z]", line):
            if not re.search(r"\d\s*$", line) and ".." not in line:
                pending_title = line
                continue
        pending_title = None

    return entries


def _norm_heading(s: str) -> str:
    return " ".join(s.split())


def _combine_title_and_body(title: str, body: str, *, dedupe_first_line: bool) -> str:
    title = title.strip()
    body = body.strip()
    if not body:
        return title
    if dedupe_first_line:
        lines = body.splitlines()
        if lines and _norm_heading(lines[0]).casefold() == _norm_heading(title).casefold():
            body = "\n".join(lines[1:]).strip()
    if not body:
        return title
    return f"{title}\n\n{body}".strip()


def _next_larger_start_page(entries: list[tuple[str, int]], i: int, n_pages: int) -> int:
    """下一条「严格更大」的起始页；用于相邻两条目录页码相同或倒序时避免正文区间为空。"""
    p0 = entries[i][1]
    for j in range(i + 1, len(entries)):
        if entries[j][1] > p0:
            return entries[j][1]
    return n_pages + 1


def build_toc_chunks(
    pages: list[str],
    entries: list[tuple[str, int]],
    *,
    chunk_id_start: int,
    content_first_page_1based: int,
    source_type: str,
    document_name: str,
    dedupe_title_line: bool,
) -> list[dict]:
    """按目录节切页；只拼接 >= content_first_page_1based 的页；丢弃节内落在该阈值之前的页。"""
    n_pages = len(pages)
    if not entries:
        return []
    chunks: list[dict] = []
    next_id = chunk_id_start
    for i, (title, p_start) in enumerate(entries):
        p_next_raw = entries[i + 1][1] if i + 1 < len(entries) else n_pages + 1
        if p_next_raw <= p_start:
            p_next = _next_larger_start_page(entries, i, n_pages)
        else:
            p_next = p_next_raw
        start_idx = max(0, min(p_start - 1, n_pages))
        end_excl = max(start_idx, min(p_next - 1, n_pages))
        body_parts: list[str] = []
        first_body_page: int | None = None
        for idx in range(start_idx, end_excl):
            page_1 = idx + 1
            if page_1 < content_first_page_1based:
                continue
            body_parts.append(pages[idx])
            if first_body_page is None:
                first_body_page = page_1
        body = "\n".join(body_parts).strip()
        if not body:
            continue
        text = _combine_title_and_body(title, body, dedupe_first_line=dedupe_title_line)
        chunks.append(
            {
                "chunk_id": next_id,
                "text": text,
                "source_type": source_type,
                "document_name": document_name,
                "section_title": title,
                "section_toc_start_page_1based": p_start,
                "section_body_first_page_1based": first_body_page,
            }
        )
        next_id += 1
    return chunks


def iter_print_chunks(chunks: Iterable[dict]) -> None:
    for c in chunks:
        print("-" * 60)
        print(f"Chunk {c['chunk_id']} | {c['source_type']} | {c['document_name']}")
        print(c["text"])
        print()


def main() -> int:
    if not PDF_PATH.is_file():
        print(f"[错误] 找不到 PDF：{PDF_PATH.resolve()}", flush=True)
        print("请在文件顶部把 PDF_PATH 改成你的 .pdf 绝对路径。", flush=True)
        return 1

    pages = read_pdf_pages(
        PDF_PATH, engine=PDF_ENGINE, skip_last_pages=SKIP_LAST_PAGES
    )
    if not pages:
        print("[错误] PDF 无有效页面（检查 SKIP_LAST_PAGES 是否过大）。", flush=True)
        return 1
    toc_blob = _toc_pages_text(pages, TOC_PARSE_PAGE_START, TOC_PARSE_PAGE_END)
    entries = parse_toc_entries(toc_blob)
    if TOC_DEBUG:
        print(
            f"[TOC_DEBUG] 目录页原文长度={len(toc_blob)} 解析条目数={len(entries)}",
            file=sys.stderr,
            flush=True,
        )
        for j, (t, pg) in enumerate(entries[:8]):
            print(f"  [{j}] p{pg}: {t[:70]}…", file=sys.stderr, flush=True)
        if len(entries) > 8:
            print(f"  … 共 {len(entries)} 条", file=sys.stderr, flush=True)
    if not entries:
        print(
            "[错误] 在指定目录页未解析到任何「标题 …… 页码」行。\n"
            f"  已扫描第 {TOC_PARSE_PAGE_START}–{TOC_PARSE_PAGE_END} 页（1-based）。\n"
            "  请确认该行格式为：标题 + 多个点 + 整数页码；或扩大 TOC_PARSE_PAGE_END。\n"
            "  扫描到的原文片段（前 800 字）：\n",
            flush=True,
        )
        print(toc_blob[:800], flush=True)
        return 1
    chunks = build_toc_chunks(
        pages,
        entries,
        chunk_id_start=TOC_CHUNKS_START_ID,
        content_first_page_1based=CONTENT_FIRST_PAGE_1BASED,
        source_type=TOC_SOURCE_TYPE,
        document_name=TOC_DOCUMENT_NAME,
        dedupe_title_line=DEDUPE_TITLE_FIRST_LINE_IN_BODY,
    )
    if not chunks:
        print(
            "[错误] 按当前 CONTENT_FIRST_PAGE_1BASED 与目录页码，没有拼出任何带正文的 chunk。\n"
            "  请检查目录里的起始页是否都落在正文起始页之前。",
            flush=True,
        )
        return 1

    if OUT_JSONL is not None:
        with OUT_JSONL.open("w", encoding="utf-8") as f:
            for c in chunks:
                out = {k: v for k, v in c.items() if not k.startswith("_")}
                f.write(json.dumps(out, ensure_ascii=False) + "\n")
        print(
            f"[完成] 目录条目 {len(entries)} → 有效 chunk {len(chunks)} 条 → {OUT_JSONL.resolve()}",
            flush=True,
        )
    else:
        if len(entries) != len(chunks):
            print(
                f"[提示] 目录解析 {len(entries)} 条，因空节（仅落在正文起始页之前等）实际输出 {len(chunks)} 个 chunk。",
                flush=True,
            )
        iter_print_chunks(chunks)

    return 0


if __name__ == "__main__":
    _code = main()
    if _code:
        sys.exit(_code)


[提示] 目录解析 34 条，因空节（仅落在正文起始页之前等）实际输出 31 个 chunk。
------------------------------------------------------------
Chunk 26 | macro_policy | Action_plan
Ensure that Frontier AI Protects Free Speech and American Values

AMERICA’S AI ACTION PLAN
Introduction
The United States is in a race to achieve global dominance in artificial intelligence (AI).
Whoever has the largest AI ecosystem will set global AI standards and reap broad economic
and military benefits. Just like we won the space race, it is imperative that the United States
and its allies win this race. President Trump took decisive steps toward achieving this goal
during his first days in office by signing Executive Order 14179, “Removing Barriers to American
Leadership in Artificial Intelligence,” calling for America to retain dominance in this global race
and directing the creation of an AI Action Plan.1
Winning the AI race will usher in a new golden age of human flourishing, economic
competitiveness, and national security for the Am

In [14]:
PDF_PATH = Path.home() / "Desktop" /  "Neural_network_(machine_learning).pdf"

# 只读到第几页（1-based，含）
MAX_PAGE_1BASED = 23

# 每块大约多少个**非空格字符**（不含空格、制表符、换行等）
TARGET_NONSPACE_CHARS = 2500

CHUNK_START_ID = 57
SOURCE_TYPE = "Wiki"
DOCUMENT_NAME = "Neural network"

PDF_ENGINE: str = "auto"  # "auto" | "pypdf" | "pdfplumber"
OUT_JSONL: Path | None = None


def _pages_without_tail(page_list: list, *, skip_last: int) -> list:
    if skip_last <= 0:
        return page_list
    n = len(page_list)
    if n <= skip_last:
        return []
    return page_list[: n - skip_last]


def _read_pages_pypdf(path: Path, *, first_n_pages: int | None) -> list[str]:
    from pypdf import PdfReader

    reader = PdfReader(str(path))
    pages = list(reader.pages)
    if first_n_pages is not None:
        pages = pages[:first_n_pages]
    return [(p.extract_text() or "") for p in pages]


def _read_pages_pdfplumber(path: Path, *, first_n_pages: int | None) -> list[str]:
    import pdfplumber

    with pdfplumber.open(str(path)) as pdf:
        pages = list(pdf.pages)
        if first_n_pages is not None:
            pages = pages[:first_n_pages]
        return [(p.extract_text() or "") for p in pages]


def read_pdf_pages_first_n(
    path: Path, *, engine: str, first_n_pages: int
) -> list[str]:
    """只读前 ``first_n_pages`` 页（1-based 即前 N 页）。"""
    eng = engine.strip().lower()
    if eng == "pypdf":
        return _read_pages_pypdf(path, first_n_pages=first_n_pages)
    if eng == "pdfplumber":
        return _read_pages_pdfplumber(path, first_n_pages=first_n_pages)
    if eng == "auto":
        try:
            return _read_pages_pdfplumber(path, first_n_pages=first_n_pages)
        except ImportError:
            return _read_pages_pypdf(path, first_n_pages=first_n_pages)
    raise ValueError(f"engine 必须是 pypdf|pdfplumber|auto，收到：{engine!r}")


def _non_space_count(s: str) -> int:
    return sum(1 for c in s if not c.isspace())


_TOKEN_RE = re.compile(r"\S+|\s+")


def split_text_by_non_space_budget(text: str, budget: int) -> list[str]:
    """
    按「非空格字符数」预算切块；在空白边界合并 token，单 token 超长则按字符硬切。
    """
    text = text.strip()
    if not text:
        return []
    if budget < 1:
        budget = 1

    parts: list[str] = []
    buf: list[str] = []
    used = 0

    def flush() -> None:
        nonlocal buf, used
        if buf:
            parts.append("".join(buf))
            buf = []
            used = 0

    for m in _TOKEN_RE.finditer(text):
        tok = m.group(0)
        if tok.isspace():
            buf.append(tok)
            continue
        add_ns = _non_space_count(tok)
        if add_ns > budget:
            flush()
            # 硬切超长「词」
            start = 0
            while start < len(tok):
                piece = []
                ns = 0
                while start < len(tok) and ns < budget:
                    ch = tok[start]
                    piece.append(ch)
                    if not ch.isspace():
                        ns += 1
                    start += 1
                parts.append("".join(piece))
            continue
        if used + add_ns > budget and used > 0:
            flush()
        buf.append(tok)
        used += add_ns

    flush()
    return [p for p in parts if p.strip()]


def build_fixed_chunks(
    full_text: str,
    *,
    budget: int,
    chunk_id_start: int,
    source_type: str,
    document_name: str,
) -> list[dict]:
    pieces = split_text_by_non_space_budget(full_text, budget)
    return [
        {
            "chunk_id": chunk_id_start + i,
            "text": p.strip(),
            "source_type": source_type,
            "document_name": document_name,
            "non_space_len": _non_space_count(p),
        }
        for i, p in enumerate(pieces)
    ]


def iter_print_chunks(chunks: Iterable[dict]) -> None:
    for c in chunks:
        print("-" * 60)
        print(
            f"Chunk {c['chunk_id']} | {c['source_type']} | {c['document_name']} "
            f"| non_space={c.get('non_space_len', '')}"
        )
        body = c["text"]
        print(body[:4000])
        if len(body) > 4000:
            print("\n… [打印截断，完整见 jsonl 或改脚本]")
        print()


def main() -> int:
    if not PDF_PATH.is_file():
        print(f"[错误] 找不到 PDF：{PDF_PATH.resolve()}", flush=True)
        return 1

    pages = read_pdf_pages_first_n(
        PDF_PATH, engine=PDF_ENGINE, first_n_pages=MAX_PAGE_1BASED
    )
    raw = "\n\n".join(pages).strip()
    if not raw:
        print("[错误] 前 N 页未抽出任何文字（扫描件？）。", flush=True)
        return 1

    chunks = build_fixed_chunks(
        raw,
        budget=TARGET_NONSPACE_CHARS,
        chunk_id_start=CHUNK_START_ID,
        source_type=SOURCE_TYPE,
        document_name=DOCUMENT_NAME,
    )
    if not chunks:
        print("[错误] 未生成任何 chunk。", flush=True)
        return 1

    if OUT_JSONL is not None:
        with OUT_JSONL.open("w", encoding="utf-8") as f:
            for c in chunks:
                f.write(json.dumps(c, ensure_ascii=False) + "\n")
        print(
            f"[完成] {len(chunks)} 个 chunk（每块约 {TARGET_NONSPACE_CHARS} 个非空格字）→ "
            f"{OUT_JSONL.resolve()}",
            flush=True,
        )
    else:
        print(
            f"[完成] {len(chunks)} 个 chunk，每块非空格字数约 {TARGET_NONSPACE_CHARS}",
            flush=True,
        )
        iter_print_chunks(chunks)

    return 0


if __name__ == "__main__":
    code = main()
    if code:
        sys.exit(code)


[完成] 22 个 chunk，每块非空格字数约 2500
------------------------------------------------------------
Chunk 57 | Wiki | Neural network | non_space=2498
Neural network (machine learning)
In machine learning, a neural network (NN) or neural
net, also known as an artificial neural network
(ANN), is a computational model inspired by the
structure and functions of biological neural
networks.[1][2]
A neural network consists of connected units or nodes
called artificial neurons, which loosely model the
neurons in the brain. Artificial neuron models that
mimic biological neurons more closely have also been
recently investigated and shown to significantly
improve performance. These are connected by edges,
which model the synapses in the brain. Each artificial
neuron receives signals from connected neurons, then
processes them and sends a signal to other connected
neurons. The "signal" is a real number, and the output
of each neuron is computed by some non-linear
function of the totality of its inputs, cal

In [19]:
PDF_PATH = Path.home() / "Desktop" /  "Climate_change.pdf"

# 只读到第几页（1-based，含）
MAX_PAGE_1BASED = 28

# 每块大约多少个**非空格字符**（不含空格、制表符、换行等）
TARGET_NONSPACE_CHARS = 2500

CHUNK_START_ID = 79
SOURCE_TYPE = "Wiki"
DOCUMENT_NAME = "Climate_change"

PDF_ENGINE: str = "auto"  # "auto" | "pypdf" | "pdfplumber"
OUT_JSONL: Path | None = None


def _pages_without_tail(page_list: list, *, skip_last: int) -> list:
    if skip_last <= 0:
        return page_list
    n = len(page_list)
    if n <= skip_last:
        return []
    return page_list[: n - skip_last]


def _read_pages_pypdf(path: Path, *, first_n_pages: int | None) -> list[str]:
    from pypdf import PdfReader

    reader = PdfReader(str(path))
    pages = list(reader.pages)
    if first_n_pages is not None:
        pages = pages[:first_n_pages]
    return [(p.extract_text() or "") for p in pages]


def _read_pages_pdfplumber(path: Path, *, first_n_pages: int | None) -> list[str]:
    import pdfplumber

    with pdfplumber.open(str(path)) as pdf:
        pages = list(pdf.pages)
        if first_n_pages is not None:
            pages = pages[:first_n_pages]
        return [(p.extract_text() or "") for p in pages]


def read_pdf_pages_first_n(
    path: Path, *, engine: str, first_n_pages: int
) -> list[str]:
    """只读前 ``first_n_pages`` 页（1-based 即前 N 页）。"""
    eng = engine.strip().lower()
    if eng == "pypdf":
        return _read_pages_pypdf(path, first_n_pages=first_n_pages)
    if eng == "pdfplumber":
        return _read_pages_pdfplumber(path, first_n_pages=first_n_pages)
    if eng == "auto":
        try:
            return _read_pages_pdfplumber(path, first_n_pages=first_n_pages)
        except ImportError:
            return _read_pages_pypdf(path, first_n_pages=first_n_pages)
    raise ValueError(f"engine 必须是 pypdf|pdfplumber|auto，收到：{engine!r}")


def _non_space_count(s: str) -> int:
    return sum(1 for c in s if not c.isspace())


_TOKEN_RE = re.compile(r"\S+|\s+")


def split_text_by_non_space_budget(text: str, budget: int) -> list[str]:
    """
    按「非空格字符数」预算切块；在空白边界合并 token，单 token 超长则按字符硬切。
    """
    text = text.strip()
    if not text:
        return []
    if budget < 1:
        budget = 1

    parts: list[str] = []
    buf: list[str] = []
    used = 0

    def flush() -> None:
        nonlocal buf, used
        if buf:
            parts.append("".join(buf))
            buf = []
            used = 0

    for m in _TOKEN_RE.finditer(text):
        tok = m.group(0)
        if tok.isspace():
            buf.append(tok)
            continue
        add_ns = _non_space_count(tok)
        if add_ns > budget:
            flush()
            # 硬切超长「词」
            start = 0
            while start < len(tok):
                piece = []
                ns = 0
                while start < len(tok) and ns < budget:
                    ch = tok[start]
                    piece.append(ch)
                    if not ch.isspace():
                        ns += 1
                    start += 1
                parts.append("".join(piece))
            continue
        if used + add_ns > budget and used > 0:
            flush()
        buf.append(tok)
        used += add_ns

    flush()
    return [p for p in parts if p.strip()]


def build_fixed_chunks(
    full_text: str,
    *,
    budget: int,
    chunk_id_start: int,
    source_type: str,
    document_name: str,
) -> list[dict]:
    pieces = split_text_by_non_space_budget(full_text, budget)
    return [
        {
            "chunk_id": chunk_id_start + i,
            "text": p.strip(),
            "source_type": source_type,
            "document_name": document_name,
            "non_space_len": _non_space_count(p),
        }
        for i, p in enumerate(pieces)
    ]


def iter_print_chunks(chunks: Iterable[dict]) -> None:
    for c in chunks:
        print("-" * 60)
        print(
            f"Chunk {c['chunk_id']} | {c['source_type']} | {c['document_name']} "
            f"| non_space={c.get('non_space_len', '')}"
        )
        body = c["text"]
        print(body[:4000])
        if len(body) > 4000:
            print("\n… [打印截断，完整见 jsonl 或改脚本]")
        print()


def main() -> int:
    if not PDF_PATH.is_file():
        print(f"[错误] 找不到 PDF：{PDF_PATH.resolve()}", flush=True)
        return 1

    pages = read_pdf_pages_first_n(
        PDF_PATH, engine=PDF_ENGINE, first_n_pages=MAX_PAGE_1BASED
    )
    raw = "\n\n".join(pages).strip()
    if not raw:
        print("[错误] 前 N 页未抽出任何文字（扫描件？）。", flush=True)
        return 1

    chunks = build_fixed_chunks(
        raw,
        budget=TARGET_NONSPACE_CHARS,
        chunk_id_start=CHUNK_START_ID,
        source_type=SOURCE_TYPE,
        document_name=DOCUMENT_NAME,
    )
    if not chunks:
        print("[错误] 未生成任何 chunk。", flush=True)
        return 1

    if OUT_JSONL is not None:
        with OUT_JSONL.open("w", encoding="utf-8") as f:
            for c in chunks:
                f.write(json.dumps(c, ensure_ascii=False) + "\n")
        print(
            f"[完成] {len(chunks)} 个 chunk（每块约 {TARGET_NONSPACE_CHARS} 个非空格字）→ "
            f"{OUT_JSONL.resolve()}",
            flush=True,
        )
    else:
        print(
            f"[完成] {len(chunks)} 个 chunk，每块非空格字数约 {TARGET_NONSPACE_CHARS}",
            flush=True,
        )
        iter_print_chunks(chunks)

    return 0


if __name__ == "__main__":
    code = main()
    if code:
        sys.exit(code)


[完成] 26 个 chunk，每块非空格字数约 2500
------------------------------------------------------------
Chunk 79 | Wiki | Climate_change | non_space=2494
Climate change
Present-day climate change includes both global
warming—the ongoing increase in global average
temperature—and its wider effects on Earth's climate
system. Climate change in a broader sense also
includes previous long-term changes to Earth's
climate. The modern-day rise in global temperatures
is driven by human activities, especially fossil fuel
(coal, oil and natural gas) burning since the Industrial
Revolution.[3][4] Fossil fuel use, deforestation, and
some agricultural and industrial practices release
greenhouse gases.[5] These gases absorb some of the
heat that the Earth radiates after it warms from
sunlight, warming the lower atmosphere. Earth's
atmosphere now has roughly 50% more carbon
Changes in surface air temperature over the past
dioxide, the main gas driving global warming, than it 50 years.[1] The Arctic has warmed the 